# Ejercicio 2: Escalamiento de tickets de soporte técnico

### Ficha PEAS
| Elemento | Descripción |
| :--- | :--- |
| **P (Performance / Desempeño)** | Minimizar tiempos de resolución, cumplir Acuerdos de Nivel de Servicio (SLA) y balancear la carga técnica. |
| **E (Environment / Entorno)** | Mesa de ayuda (Help Desk), cola de incidentes y red de usuarios/clientes de soporte. |
| **A (Actuators / Acciones)** | Enrutar el ticket: "atender en nivel 1", "escalar a nivel 2", o "escalar a nivel 3". |
| **S (Sensors / Percepciones)** | Parámetros del ticket: `tiempo_espera_minutos` (entero), `nivel_urgencia` ("baja", "media", "alta") y `cliente_premium` (booleano). |
| **Objetivo** | Asignar cada caso al nivel técnico idóneo evitando la saturación del soporte especializado. |

### Justificación de las reglas
Un reporte de urgencia alta compromete la operatividad del cliente y demanda atención inmediata en Nivel 3 para contener el impacto del incidente. En urgencias medias o bajas, la condición premium y el tiempo acumulado protegen los SLA: los clientes prioritarios escalan a Nivel 2 o 3 si superan tiempos prudentes (20 o 60 minutos). Por último, las incidencias rutinarias sin riesgo de incumplimiento se resuelven en Nivel 1, protegiendo a los ingenieros avanzados de sobrecargas operativas.

In [1]:
def agente_soporte(tiempo_espera_minutos, nivel_urgencia, cliente_premium):
    """
    Agente para enrutar tickets según criticidad, SLA y prioridad de cuenta.
    Retorna: (accion, motivo)
    """
    urgencia = nivel_urgencia.strip().lower()
    
    # Urgencia alta: atención máxima inmediata
    if urgencia == "alta":
        return "escalar a nivel 3", "urgencia alta reportada requiere atencion especializada inmediata"
        
    # Urgencia media: ponderación por SLA y categoría de cuenta
    elif urgencia == "media":
        if cliente_premium and tiempo_espera_minutos > 20:
            return "escalar a nivel 3", f"cliente premium con {tiempo_espera_minutos} min de espera en urgencia media"
        elif tiempo_espera_minutos > 45:
            return "escalar a nivel 2", f"tiempo de espera ({tiempo_espera_minutos} min) excede umbral estandar de SLA"
        elif cliente_premium:
            return "escalar a nivel 2", "cliente premium priorizado para resolucion intermedia"
        else:
            return "atender en nivel 1", f"urgencia media dentro de parametros ordinarios ({tiempo_espera_minutos} min)"
            
    # Urgencia baja
    elif urgencia == "baja":
        if cliente_premium and tiempo_espera_minutos > 60:
            return "escalar a nivel 2", f"cliente premium con demora excesiva ({tiempo_espera_minutos} min)"
        elif tiempo_espera_minutos > 120:
            return "escalar a nivel 2", f"retraso atipico acumulado ({tiempo_espera_minutos} min) en caso simple"
        else:
            return "atender en nivel 1", "incidencia de baja urgencia asignada al flujo regular de Nivel 1"
            
    else:
        return "atender en nivel 1", "nivel de urgencia no catalogado; derivado a recepcion Nivel 1"

In [2]:
pruebas_soporte = [
    (10, "alta", False),   # Urgencia alta directa -> Nivel 3
    (25, "media", True),   # Media + Premium + espera moderada -> Nivel 3
    (15, "media", True),   # Media + Premium + espera baja -> Nivel 2
    (50, "media", False),  # Media + Estandar + espera alta -> Nivel 2
    (10, "media", False),  # Media + Estandar + espera normal -> Nivel 1
    (90, "baja", True),    # Baja + Premium + espera prolongada -> Nivel 2
    (15, "baja", False)    # Baja + Estandar -> Nivel 1
]

print(f"{'Espera (min)':<14} | {'Urgencia':<10} | {'Premium':<8} | {'Acción':<22} | {'Motivo'}")
print("-" * 90)
for espera, urg, prem in pruebas_soporte:
    accion, motivo = agente_soporte(espera, urg, prem)
    print(f"{espera:<14} | {urg:<10} | {str(prem):<8} | {accion:<22} | {motivo}")

Espera (min)   | Urgencia   | Premium  | Acción                 | Motivo
------------------------------------------------------------------------------------------
10             | alta       | False    | escalar a nivel 3      | urgencia alta reportada requiere atencion especializada inmediata
25             | media      | True     | escalar a nivel 3      | cliente premium con 25 min de espera en urgencia media
15             | media      | True     | escalar a nivel 2      | cliente premium priorizado para resolucion intermedia
50             | media      | False    | escalar a nivel 2      | tiempo de espera (50 min) excede umbral estandar de SLA
10             | media      | False    | atender en nivel 1     | urgencia media dentro de parametros ordinarios (10 min)
90             | baja       | True     | escalar a nivel 2      | cliente premium con demora excesiva (90 min)
15             | baja       | False    | atender en nivel 1     | incidencia de baja urgencia asignada al fl